# 01 — Scraping
Collect civil engineering x AI articles from web searches and (optionally) NewsAPI.

- Update CE/AI keyword lists as needed.
- Set environment variable `NEWSAPI_KEY` to enable NewsAPI results; otherwise only HTML scraping runs.
- Output saved to `data/raw/articles.csv`.



In [1]:
import os
import sys
from pathlib import Path

# Load .env if available for API keys (NEWSAPI_KEY, MEDIASTACK_KEY, BING_API_KEY, SERPAPI_KEY)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Optional: set keys inline here for notebook-only use (remove quotes and fill values)
os.environ["NEWSAPI_KEY"] = ""
os.environ["MEDIASTACK_KEY"] = ""
os.environ["BING_API_KEY"] = ""
os.environ["SERPAPI_KEY"] = ""

# Drop any empty-string keys so toggles remain off when blanks are used
for _k in ["NEWSAPI_KEY", "MEDIASTACK_KEY", "BING_API_KEY", "SERPAPI_KEY"]:
    if os.getenv(_k) == "":
        os.environ.pop(_k, None)

# Ensure project root (containing src/) is on sys.path
root = Path().resolve()
if not (root / "src").exists():
    root = root.parent
sys.path.append(str(root))

from src.scraping.aggregator.aggregate_all import collect_all_articles

# Keyword matrices
ce_terms = [
    "structural",
    "geotechnical",
    "transportation",
    "construction management",
    "environmental",
]
ai_terms = [
    "ai",
    "machine learning",
    "computer vision",
    "predictive",
    "robotics",
]

# Runtime knobs (balanced for ~1k articles <20 min on laptop)
max_api_results = 400        # spread across query_terms
num_google_pages = 3         # per query
site_page_limit = 3          # per site scraper per query
include_legacy_html = True   # uses earlier HTML scraper + NewsAPI if key present

(root / "data" / "raw").mkdir(parents=True, exist_ok=True)

print("Working directory:", root)
print({k: bool(os.getenv(k)) for k in ["NEWSAPI_KEY", "MEDIASTACK_KEY", "BING_API_KEY", "SERPAPI_KEY"]})

master_df = collect_all_articles(
    ce_terms,
    ai_terms,
    max_api_results=max_api_results,
    num_google_pages=num_google_pages,
    site_page_limit=site_page_limit,
    include_legacy_html=include_legacy_html,
)

print("Articles collected:", len(master_df))
master_df.head()


Working directory: C:\Users\Asus\Desktop\CE 49X\ce49x-final
{'NEWSAPI_KEY': False, 'MEDIASTACK_KEY': False, 'BING_API_KEY': False, 'SERPAPI_KEY': False}
Articles collected: 1038


,title,date,source,url,full_text,source_type,raw_source
0,7 construction project milestones from Novembe...,"Wed, 10 Dec 2025 15:47:28 -0500",Construction Dive - Latest News,https://www.constructiondive.com/news/7-constr...,"<figure><div><img src=""https://imgproxy.divecd...",rss,https://www.constructiondive.com/feeds/news/
1,Fed’s rate cut boosts existing construction pr...,"Wed, 10 Dec 2025 14:13:00 -0500",Construction Dive - Latest News,https://www.constructiondive.com/news/feds-rat...,"<figure><div><img src=""https://imgproxy.divecd...",rss,https://www.constructiondive.com/feeds/news/
2,AI nears ‘tipping point’ in construction as co...,"Wed, 10 Dec 2025 11:40:00 -0500",Construction Dive - Latest News,https://www.constructiondive.com/news/builders...,"<figure><div><img src=""https://imgproxy.divecd...",rss,https://www.constructiondive.com/feeds/news/
3,"Facing extreme rainfall and flooding, NYC is t...","Wed, 10 Dec 2025 09:50:00 -0500",Construction Dive - Latest News,https://www.constructiondive.com/news/bluebelt...,"<figure><div><img src=""https://imgproxy.divecd...",rss,https://www.constructiondive.com/feeds/news/
4,Garco lands $200M semiconductor expansion project,"Tue, 09 Dec 2025 16:41:00 -0500",Construction Dive - Latest News,https://www.constructiondive.com/news/garco-co...,"<figure><div><img src=""https://imgproxy.divecd...",rss,https://www.constructiondive.com/feeds/news/


In [2]:
# Save master file and show counts by source/source_type
from collections import Counter

out_path = Path("data/raw/articles_master.csv")
master_df.to_csv(out_path, index=False)

print("Saved to:", out_path)
print("By source_type:", Counter(master_df["source_type"]))
print("Top sources:", master_df["source"].value_counts().head(10))


Saved to: data\raw\articles_master.csv
By source_type: Counter({'site': 647, 'api': 276, nan: 75, 'rss': 40})
Top sources: source
NewCivilEngineer                             387
GDELT                                        276
ConstructionDive                             252
www.constructiondive.com                      75
Artificial Intelligence (AI) | TechCrunch     20
Engineering.com                               18
Construction Dive - Latest News               10
Name: count, dtype: int64
